# NLI Lyrics Corpus Builder (v2)

**Goal:** Build a corpus of English-language songs by artists with a known L1, for Native Language Identification research.

**Scope:** Phases 2–6 of the data collection plan.
- **Phase 2** — Build candidate artist pool (Wikidata + MusicBrainz)
- **Phase 3** — Pre-filter candidates (genre, Wikipedia presence, optional year)
- **Phase 4** — Scrape lyrics from Genius (with error handling)
- **Phase 5** — Language filtering with fastText
- **Phase 6** — Metadata filtering (covers, duplicates)

Manual L1 verification (Phase 7) happens after this notebook on the produced `artists_for_verification.csv`.

**New additions (v2)**
- Tightened genre filter: dropped generic `latin` keyword (was matching bolero/folk artists who don't write in English)
- Added optional year filter to drop pre-1970s artists
- Scrape function wrapped in try/except so one bad artist doesn't kill the whole run
- Plain `tqdm` (not `tqdm.notebook`) to avoid `ipywidgets` dependency issues

## Setup

### Install dependencies

Run once:

```bash
pip install lyricsgenius requests musicbrainzngs pandas fasttext-wheel tqdm python-dotenv
```

### API credentials

You need a **Genius API token** (required for Phase 4). Get one here:
- Go to https://genius.com/api-clients
- Sign up / log in, click "New API Client"
- Fill in any app name and website (e.g., `http://example.com`)
- Copy the **Client Access Token**

Create a `.env` file in the same folder as this notebook:

```
GENIUS_TOKEN=your_token_here
```

**Make sure `.env` is in your `.gitignore`** before committing anything.

In [1]:
import os
import json
import time
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import requests
from tqdm import tqdm  # Plain tqdm (no ipywidgets dependency)
from dotenv import load_dotenv

load_dotenv()

GENIUS_TOKEN = os.getenv("GENIUS_TOKEN")
assert GENIUS_TOKEN, "Set GENIUS_TOKEN in .env before running Phase 4"

print("Genius token loaded")

/Users/nataliajimenez/miniforge3/envs/auc/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Genius token loaded


## Configuration — change this block for each L1

Everything downstream reads from `CONFIG`. To swap languages, edit this cell and re-run.

In [25]:
CONFIG = {
    # L1 label used in output files
    "L1": "spanish",

    # ISO 3166-1 alpha-2 country codes where this L1 is dominant
    "countries": [
        "ES", "MX", "AR", "CO", "PE", "CL", "VE", "EC", "GT", "CU",
        "DO", "HN", "BO", "SV", "PY", "NI", "CR", "PR", "PA", "UY",
    ],

    # fastText language code for this L1 (for detecting/excluding it in lyrics)
    "l1_fasttext_code": "es",

    # Wikidata occupation QIDs to include
    # rapper, hip-hop musician, singer, singer-songwriter
    "occupation_qids": ["Q2252262", "Q177220", "Q753110", "Q488205"],

    # Genre keywords for Phase 3 filter
    # Note: "latin" deliberately excluded, it matches all Spanish-language genres
    # including bolero, ranchera, etc., which rarely have English releases
    "target_genres": [
        "hip hop", "hip-hop", "rap", "trap", "reggaeton",
        "urban", "r&b", "rnb", "drill", "grime", "dembow",
    ],

    # Optional: exclude artists born before this year (None to disable)
    # 1970 cuts most pre-modern-pop artists who didn't write English-language tracks
    "min_birth_year": 1970,

    # Phase 5 thresholds
    "min_english_ratio": 0.5,
    "borderline_ratio": 0.3,
    "min_lines_per_song": 8,

    # Phase 8 requirement (enforced at assembly)
    "min_songs_per_artist": 3,

    # Output structure
    "data_dir": Path("./data"),
}

DATA = CONFIG["data_dir"] / CONFIG["L1"]
for sub in ["raw", "interim", "processed", "logs"]:
    (DATA / sub).mkdir(parents=True, exist_ok=True)

print(f"Workspace: {DATA.resolve()}")
print(f"L1: {CONFIG['L1']} | Countries: {len(CONFIG['countries'])} | fastText code: {CONFIG['l1_fasttext_code']}")

Workspace: /Users/nataliajimenez/Desktop/AUC/TM/project/data/spanish
L1: spanish | Countries: 20 | fastText code: es


---
# Phase 2 — Candidate artist pool

## 2A — Wikidata SPARQL

In [ ]:
def iso_to_wikidata_country(iso_codes):
    """Map ISO country codes to Wikidata QIDs."""
    mapping = {
        "ES": "Q29",  "MX": "Q96",  "AR": "Q414", "CO": "Q739", "PE": "Q419",
        "CL": "Q298", "VE": "Q717", "EC": "Q736", "GT": "Q774", "CU": "Q241",
        "DO": "Q786", "HN": "Q783", "BO": "Q750", "SV": "Q792", "PY": "Q733",
        "NI": "Q811", "CR": "Q800", "PR": "Q1183","PA": "Q804", "UY": "Q77",
        "IT": "Q38",  "NL": "Q55",  "BE": "Q31",  "RU": "Q159", "BY": "Q184",
        "UA": "Q212", "KZ": "Q232", "US": "Q30",  "GB": "Q145", "CA": "Q16",
        "AU": "Q408", "IE": "Q27",  "NZ": "Q664",
    }
    return [mapping[c] for c in iso_codes if c in mapping]


def build_wikidata_query(country_qids, occupation_qids):
    countries_block = " ".join(f"wd:{q}" for q in country_qids)
    occupations_block = " ".join(f"wd:{q}" for q in occupation_qids)
    return f"""
    SELECT DISTINCT ?artist ?artistLabel ?birthPlaceLabel ?birthYear ?genreLabel ?enwiki ?genius
    WHERE {{
      ?artist wdt:P31 wd:Q5 ;
              wdt:P106 ?occupation ;
              wdt:P27 ?country .
      VALUES ?occupation {{ {occupations_block} }}
      VALUES ?country {{ {countries_block} }}
      OPTIONAL {{ ?artist wdt:P19 ?birthPlace . }}
      OPTIONAL {{ ?artist wdt:P569 ?birthDate . BIND(YEAR(?birthDate) AS ?birthYear) }}
      OPTIONAL {{ ?artist wdt:P136 ?genre . }}
      OPTIONAL {{ ?artist wdt:P2373 ?genius . }}
      OPTIONAL {{
        ?enwiki schema:about ?artist ;
                schema:isPartOf <https://en.wikipedia.org/> .
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en" . }}
    }}
    LIMIT 5000
    """


def query_wikidata(sparql_query):
    url = "https://query.wikidata.org/sparql"
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": "NLI-research-notebook/1.0 (academic, AUC text mining course)",
    }
    response = requests.get(url, params={"query": sparql_query}, headers=headers, timeout=90)
    response.raise_for_status()
    return response.json()


country_qids = iso_to_wikidata_country(CONFIG["countries"])
query = build_wikidata_query(country_qids, CONFIG["occupation_qids"])
print(f"Querying Wikidata for {len(country_qids)} countries × {len(CONFIG['occupation_qids'])} occupations...")

raw = query_wikidata(query)
bindings = raw["results"]["bindings"]
print(f"Returned {len(bindings)} rows (may include duplicates due to multi-genre artists)")

In [ ]:
def parse_wikidata_results(bindings):
    rows = []
    for b in bindings:
        rows.append({
            "wikidata_id": b["artist"]["value"].rsplit("/", 1)[-1],
            "name": b.get("artistLabel", {}).get("value", ""),
            "birthplace": b.get("birthPlaceLabel", {}).get("value", ""),
            "birth_year": b.get("birthYear", {}).get("value", ""),
            "genre": b.get("genreLabel", {}).get("value", ""),
            "enwiki_url": b.get("enwiki", {}).get("value", ""),
            "genius_slug": b.get("genius", {}).get("value", ""),
        })
    df = pd.DataFrame(rows)
    # Collapse multi-genre duplicates: keep one row per artist, join genres
    df = (
        df.groupby(["wikidata_id", "name", "birthplace", "birth_year", "enwiki_url", "genius_slug"],
                   dropna=False, as_index=False)
          .agg({"genre": lambda s: "; ".join(sorted({g for g in s if g}))})
    )
    df["source"] = "wikidata"
    return df


candidates_wd = parse_wikidata_results(bindings)
candidates_wd.to_csv(DATA / "interim" / "candidates_wikidata.csv", index=False)
print(f"Wikidata: {len(candidates_wd)} unique artists")
candidates_wd.head(10)

## 2B — MusicBrainz

In [ ]:
import musicbrainzngs

musicbrainzngs.set_useragent("NLI-research-notebook", "1.0", "academic@auc.nl")


def fetch_musicbrainz_artists(country_codes, tags=("hip hop", "rap", "reggaeton"), limit=100):
    rows = []
    for cc in tqdm(country_codes, desc="MusicBrainz countries"):
        for tag in tags:
            query = f'country:{cc} AND tag:"{tag}"'
            try:
                result = musicbrainzngs.search_artists(query=query, limit=limit)
            except musicbrainzngs.WebServiceError as e:
                print(f"  {cc}/{tag}: error {e}")
                time.sleep(2)
                continue
            for a in result.get("artist-list", []):
                rows.append({
                    "musicbrainz_id": a.get("id"),
                    "name": a.get("name"),
                    "country": a.get("country"),
                    "tags": "; ".join(t["name"] for t in a.get("tag-list", [])) if a.get("tag-list") else "",
                    "disambiguation": a.get("disambiguation", ""),
                    "source": "musicbrainz",
                })
            time.sleep(1.0)  # MusicBrainz requires 1 req/sec
    return pd.DataFrame(rows).drop_duplicates(subset="musicbrainz_id")


candidates_mb = fetch_musicbrainz_artists(CONFIG["countries"])
candidates_mb.to_csv(DATA / "interim" / "candidates_musicbrainz.csv", index=False)
print(f"MusicBrainz: {len(candidates_mb)} unique artists")
candidates_mb.head(10)

## 2C — Merge candidates from both sources

In [ ]:
def normalize_name(s):
    if pd.isna(s):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s)
    return s


wd = candidates_wd.copy()
mb = candidates_mb.copy()
wd["name_norm"] = wd["name"].apply(normalize_name)
mb["name_norm"] = mb["name"].apply(normalize_name)

merged = wd.merge(mb, on="name_norm", how="outer", suffixes=("_wd", "_mb"))
merged["name"] = merged["name_wd"].fillna(merged["name_mb"])
merged["sources"] = merged.apply(
    lambda r: "; ".join([s for s in [r.get("source_wd"), r.get("source_mb")] if pd.notna(s)]),
    axis=1,
)

candidates = merged[["name", "name_norm", "sources", "wikidata_id", "musicbrainz_id",
                     "birthplace", "birth_year", "genre", "country", "tags",
                     "enwiki_url", "genius_slug", "disambiguation"]].drop_duplicates("name_norm")

candidates.to_csv(DATA / "interim" / "candidates_merged.csv", index=False)
print(f"Merged candidates: {len(candidates)}")
print(f"  Wikidata only:    {(merged['source_wd'].notna() & merged['source_mb'].isna()).sum()}")
print(f"  MusicBrainz only: {(merged['source_mb'].notna() & merged['source_wd'].isna()).sum()}")
print(f"  Both sources:     {(merged['source_wd'].notna() & merged['source_mb'].notna()).sum()}")

---
# Phase 3 — Pre-filter

Aggressive filtering before scraping. Each filter is named so you can see what dropped what.

**Why these filters:**
1. **Genre filter** — only keep hip-hop/rap/urban genres where English-language tracks are common. Generic "latin" tag deliberately excluded.
2. **Wikipedia filter** — require an English Wikipedia page. This is a proxy for "documented artist with verifiable bio", which we'll need anyway for L1 verification (Phase 7).
3. **Year filter (optional)** — drop pre-1970 artists who are unlikely to have English-language modern hip-hop releases.

In [3]:
# load data if not first time running
candidates = pd.read_csv(DATA / "interim" / "candidates_merged.csv")
print(f"Loaded {len(candidates)} existing candidates from disk")

Loaded 4631 existing candidates from disk


In [4]:
pre_filtered = candidates.copy()
print(f"Starting candidates:        {len(pre_filtered)}")

# Filter 1: name not empty
pre_filtered = pre_filtered[pre_filtered["name"].str.len() > 0]
print(f"After name filter:          {len(pre_filtered)}")

# Filter 2: relevant genre (hip-hop/rap/urban, not generic 'latin')
def has_target_genre(row):
    genres = " ".join(str(x).lower() for x in [row.get("genre", ""), row.get("tags", "")])
    return any(kw in genres for kw in CONFIG["target_genres"])

pre_filtered = pre_filtered[pre_filtered.apply(has_target_genre, axis=1)]
print(f"After genre filter:         {len(pre_filtered)}")

# Filter 3: must have an English Wikipedia page
pre_filtered = pre_filtered[
    pre_filtered["enwiki_url"].notna() & (pre_filtered["enwiki_url"] != "")
]
print(f"After Wikipedia filter:     {len(pre_filtered)}")

# Filter 4 (optional): birth year cutoff
if CONFIG["min_birth_year"] is not None:
    # Convert birth_year to numeric (some are strings, some empty)
    pre_filtered["birth_year_num"] = pd.to_numeric(pre_filtered["birth_year"], errors="coerce")
    # Keep rows where birth_year is unknown (NaN) OR >= cutoff
    mask = pre_filtered["birth_year_num"].isna() | (pre_filtered["birth_year_num"] >= CONFIG["min_birth_year"])
    pre_filtered = pre_filtered[mask]
    print(f"After year filter (>={CONFIG['min_birth_year']}):  {len(pre_filtered)}")

pre_filtered.to_csv(DATA / "interim" / "candidates_prefiltered.csv", index=False)

n = len(pre_filtered)
est_min = n * 30 / 60
print(f"\nEstimated scrape time: {est_min:.0f} min ({est_min/60:.1f} hours) at max_songs=25")

if n < 100:
    print("Good, proceed with scraping")
elif n < 300:
    print("Acceptable, proceed but be patient")
else:
    print(f"Still too many ({n}). Consider tightening CONFIG['target_genres'] or raising min_birth_year.")

print("\nSample of survivors (do these names look like artists who would write English songs?):")
pre_filtered[["name", "country", "birth_year", "genre", "tags"]].head(20)

Starting candidates:        4631
After name filter:          4631
After genre filter:         485
After Wikipedia filter:     70
After year filter (>=1970):  62

Estimated scrape time: 31 min (0.5 hours) at max_songs=25
✅ Good — proceed with scraping

Sample of survivors (do these names look like artists who would write English songs?):


,name,country,birth_year,genre,tags
65,Aid Alonso,NaN,1990.0,hip-hop,NaN
119,Aldo Ranks,PA,1973.0,reggae en Español,latin; latin urban; reggaeton
233,Amandititita,NaN,1979.0,cumbia; reggaeton,NaN
260,Ana Brenda Contreras,NaN,1986.0,contemporary R&B,NaN
284,Ana Tijoux,CL,1977.0,hip-hop,hip-hop; hip hop; chile; female vocals; rapper
365,"Antonio ""El Potro"" Álvarez",NaN,1979.0,Latin pop; merengue; reggaeton; tropical music,NaN
369,Antonio José,NaN,1995.0,Latin pop; ballade; flamenco; pop music; pop r...,NaN
406,Arca,ES,1989.0,experimental music; neoperreo,trip-hop; electronic; pop; industrial; ambient...
417,Arianna Puello,ES,1977.0,rapping,hip hop
462,Babo,MX,1976.0,Mexican hip-hop,hip hop


In [5]:
pre_filtered[["name", "country", "birth_year", "genre"]].to_string()

'                            name country  birth_year                                                                                                   genre\n65                    Aid Alonso     NaN      1990.0                                                                                                 hip-hop\n119                   Aldo Ranks      PA      1973.0                                                                                       reggae en Español\n233                 Amandititita     NaN      1979.0                                                                                       cumbia; reggaeton\n260         Ana Brenda Contreras     NaN      1986.0                                                                                        contemporary R&B\n284                   Ana Tijoux      CL      1977.0                                                                                                 hip-hop\n365   Antonio "El Potro" Álvarez     NaN      1979.

By looking at this output I have decided to remove artists which may add noise by having ghostwriters and artists that are not native spanish speakers (remove/change for other languages)

In [6]:
# Drop high ghostwriter risk before scraping
ghostwriter_risk = [
    "J Balvin", "Ricky Martin", "Thalía", "Paulina Rubio",
    "Tini Stoessel", "Camilo", "Danna", "Lele Pons", "Yasmin Deliz",
    "Antonio José", "Beatriz Luengo", "Fonseca", "Erika Ender",
    "Diana Neira", "Ana Brenda Contreras", "María Isabel López",
    "Mayra Goñi", "Wendy Sulca",
    # Foreign-born or US-raised, will fail L1:
    "Farid Bang", "J.R. Writer", "Zack de la Rocha",
]

before = len(pre_filtered)
pre_filtered = pre_filtered[~pre_filtered["name"].isin(ghostwriter_risk)]
print(f"Dropped {before - len(pre_filtered)} ghostwriter-risk / non-L1 artists")
print(f"Remaining: {len(pre_filtered)}")

Dropped 21 ghostwriter-risk / non-L1 artists
Remaining: 41


---
# Phase 4 — Scrape lyrics from Genius

**Checkpointed:** each artist's scrape saves to `data/<L1>/raw/<artist_slug>.json`. If the loop crashes or you interrupt it, re-running skips artists already done.

**Wrapped in try/except:** one bad artist won't kill the whole run.

Firts we perform a pre-scrapping sanity check:

In [7]:
print(f"About to scrape {len(pre_filtered)} artists")
print(f"Estimated time: {len(pre_filtered) * 30 / 60:.0f} minutes")
print("\nFirst 10:")
print(pre_filtered[["name", "country"]].head(10).to_string())
print("\nLast 10:")
print(pre_filtered[["name", "country"]].tail(10).to_string())

About to scrape 41 artists
Estimated time: 20 minutes

First 10:
                           name country
65                   Aid Alonso     NaN
119                  Aldo Ranks      PA
233                Amandititita     NaN
284                  Ana Tijoux      CL
365  Antonio "El Potro" Álvarez     NaN
406                        Arca      ES
417              Arianna Puello      ES
462                        Babo      MX
562                   Bocafloja     NaN
620                    CandyMan      CU

Last 10:
               name country
3453  Osmani García      CU
3528   Papi Sánchez      DO
3546   Pato Machete     NaN
3695          Porta     NaN
3702         P-Star     NaN
3727       Q5932808     NaN
3860         Reykon     NaN
3933          Rocca     NaN
4370      Tote King     NaN
4572  Yotuel Romero     NaN


In [8]:
import lyricsgenius

genius = lyricsgenius.Genius(
    GENIUS_TOKEN,
    remove_section_headers=True,
    skip_non_songs=True,
    excluded_terms=["(Remix)", "(Live)", "(Acoustic)", "(Demo)", "(Instrumental)"],
    timeout=30,
    retries=3,
)
genius.verbose = False  # Set as attribute, not constructor arg (compatibility fix)
print("Genius client initialized")

Genius client initialized


In [19]:
def slugify(name):
    s = re.sub(r"[^\w\s-]", "", str(name).lower()).strip()
    return re.sub(r"[-\s]+", "_", s)


def scrape_artist(name, max_songs=25):
    """Scrape discography; return dict with songs or error.
    Compatible with lyricsgenius 3.12+ which uses to_dict() for IDs.
    """
    try:
        artist = genius.search_artist(name, max_songs=max_songs, sort="popularity")
    except Exception as e:
        return {"error": str(e), "name": name}
    if artist is None:
        return {"error": "not_found", "name": name}
    
    # Get artist ID from to_dict
    try:
        artist_id = artist.to_dict().get("id")
    except Exception:
        artist_id = None
    
    songs = []
    for song in artist.songs:
        try:
            song_dict = song.to_dict()
            song_id = song_dict.get("id")
        except Exception:
            song_id = None
        
        # featured_artists handling - check if it's a list of dicts or objects
        featured = []
        try:
            for fa in (song.featured_artists or []):
                if isinstance(fa, dict):
                    featured.append(fa.get("name", ""))
                elif hasattr(fa, "name"):
                    featured.append(fa.name)
        except Exception:
            pass
        
        songs.append({
            "song_id": song_id,
            "title": song.title,
            "lyrics": song.lyrics or "",
            "url": song.url,
            "featured_artists": featured,
        })
    
    return {"name": name, "genius_id": artist_id, "songs": songs}


def scrape_all(candidates_df, max_songs=25):
    """Loop over candidates with checkpointing and error handling."""
    raw_dir = DATA / "raw"
    log_path = DATA / "logs" / "scrape_log.jsonl"

    for _, row in tqdm(candidates_df.iterrows(), total=len(candidates_df), desc="Scraping Genius"):
        slug = slugify(row["name"])
        if not slug:
            continue
        out_path = raw_dir / f"{slug}.json"
        if out_path.exists():
            continue  # Checkpoint: skip already scraped

        # Wrap entire scrape attempt so one bad artist doesn't kill the run
        try:
            result = scrape_artist(row["name"], max_songs=max_songs)
        except Exception as e:
            result = {"error": f"unexpected: {e}", "name": row["name"]}

        try:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)
            with open(log_path, "a", encoding="utf-8") as f:
                f.write(json.dumps({
                    "name": row["name"],
                    "slug": slug,
                    "status": "error" if "error" in result else "ok",
                    "n_songs": len(result.get("songs", [])),
                }) + "\n")
        except Exception as e:
            print(f"Failed to save {slug}: {e}")

        time.sleep(1.0)  # Be polite to Genius

print("Scrape function ready.")

Scrape function ready.


In [20]:
# Kick off the scrape. This can take a while - check the time estimate above.
# If you interrupt, re-running this cell resumes from where it stopped (checkpointed).
scrape_all(pre_filtered, max_songs=25)
print("Scrape complete")

Scraping Genius: 100%|██████████████████████████████████████████████████████████████████| 41/41 [39:32<00:00, 57.87s/it]

Scrape complete


In [21]:
# Load all scraped data into a song-level DataFrame
def load_scraped_songs():
    rows = []
    n_errors = 0
    for json_path in (DATA / "raw").glob("*.json"):
        with open(json_path, encoding="utf-8") as f:
            data = json.load(f)
        if "error" in data:
            n_errors += 1
            continue
        for song in data.get("songs", []):
            rows.append({
                "artist_name": data["name"],
                "artist_slug": json_path.stem,
                "genius_artist_id": data.get("genius_id"),
                "song_id": song["song_id"],
                "title": song["title"],
                "lyrics": song["lyrics"],
                "url": song["url"],
                "featured_artists": "; ".join(song["featured_artists"]),
            })
    print(f"Loaded {len(rows)} songs from successful scrapes ({n_errors} artists had errors)")
    return pd.DataFrame(rows)


songs_raw = load_scraped_songs()
print(f"Unique artists with songs: {songs_raw['artist_slug'].nunique()}")
songs_raw.head()

Loaded 638 songs from successful scrapes (1 artists had errors)
Unique artists with songs: 35


,artist_name,artist_slug,genius_artist_id,song_id,title,lyrics,url,featured_artists
0,La Materialista,la_materialista,456495,3570081,Niveles,Si la gente me tiene envidia\nNo les pares que...,https://genius.com/La-materialista-niveles-lyrics,
1,La Materialista,la_materialista,456495,1966299,La Chapa Que Vibran,"¿Aló?\nPapi, ¿te gustan la' chapa' que vibran?...",https://genius.com/La-materialista-la-chapa-qu...,
2,La Materialista,la_materialista,456495,3522039,Te Lo Compré,"Entonce', yo no soy el hombre de la casa\nSí, ...",https://genius.com/La-materialista-te-lo-compr...,Chimbala
3,La Materialista,la_materialista,456495,3103575,Taka Taka,Mami Cuidate que en la calle\nLos perros están...,https://genius.com/La-materialista-taka-taka-l...,Nfasis
4,La Materialista,la_materialista,456495,3829156,Nunca Maniqui,(Con Light G)\nYa me dieron lu' de que tú no t...,https://genius.com/La-materialista-nunca-maniq...,


---
# Phase 5 — Language filtering with fastText

In [22]:
import urllib.request

FASTTEXT_MODEL_PATH = DATA / "lid.176.ftz"
FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz"

if not FASTTEXT_MODEL_PATH.exists():
    print("Downloading fastText language model (~1MB)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_MODEL_PATH)
    print(f"Saved to {FASTTEXT_MODEL_PATH}")
else:
    print(f"Model already present at {FASTTEXT_MODEL_PATH}")

import fasttext
fasttext.FastText.eprint = lambda *a, **k: None  # Suppress deprecation warning
ft_model = fasttext.load_model(str(FASTTEXT_MODEL_PATH))
print("fastText model loaded")

Saved to data/spanish/lid.176.ftz
fastText model loaded


In [26]:
def clean_lyrics(text):
    """Light cleaning before language ID."""
    if not text:
        return ""
    text = re.sub(r"\[[^\]]+\]", "", text)  # Strip [Verse], [Chorus] etc.
    text = re.sub(r"^\d+\s+Contributors.*?Lyrics", "", text, flags=re.DOTALL)  # Genius header
    text = re.sub(r"\d*Embed\s*$", "", text)  # Genius footer
    text = re.sub(r"\s+", " ", text).strip()
    return text


def analyze_song_language(text, ft):
    """Per-song language stats based on line-level classification."""
    cleaned = clean_lyrics(text)
    lines = [ln.strip() for ln in re.split(r"[\n.!?]", cleaned) if ln.strip() and len(ln.strip()) > 3]
    if not lines:
        return {"n_lines": 0, "english_ratio": 0, "l1_ratio": 0, "top_lang": None}
    labels, _ = ft.predict(lines, k=1)
    lang_codes = [lbl[0].replace("__label__", "") for lbl in labels]
    lang_counts = Counter(lang_codes)
    total = len(lang_codes)
    return {
        "n_lines": total,
        "english_ratio": lang_counts.get("en", 0) / total,
        "l1_ratio": lang_counts.get(CONFIG["l1_fasttext_code"], 0) / total,
        "top_lang": lang_counts.most_common(1)[0][0],
    }


lang_stats = []
for _, row in tqdm(songs_raw.iterrows(), total=len(songs_raw), desc="Language ID"):
    lang_stats.append(analyze_song_language(row["lyrics"], ft_model))

lang_df = pd.DataFrame(lang_stats)
songs_with_lang = pd.concat([songs_raw.reset_index(drop=True), lang_df], axis=1)
songs_with_lang["clean_lyrics"] = songs_with_lang["lyrics"].apply(clean_lyrics)
songs_with_lang.to_csv(DATA / "interim" / "songs_with_language.csv", index=False)

print(f"\nSongs analyzed: {len(songs_with_lang)}")
print("\nTop language distribution:")
print(songs_with_lang["top_lang"].value_counts().head(10))

Language ID: 100%|██████████████████████████████████████████████████████████████████| 638/638 [00:00<00:00, 3811.59it/s]



Songs analyzed: 638

Top language distribution:
top_lang
es    580
en     25
fr     23
pl      3
pt      2
sk      1
it      1
km      1
de      1
ar      1
Name: count, dtype: int64


In [27]:
# Apply thresholds
qualifying = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

borderline = songs_with_lang[
    (songs_with_lang["english_ratio"] >= CONFIG["borderline_ratio"]) &
    (songs_with_lang["english_ratio"] < CONFIG["min_english_ratio"]) &
    (songs_with_lang["n_lines"] >= CONFIG["min_lines_per_song"])
].copy()

qualifying.to_csv(DATA / "interim" / "songs_english.csv", index=False)
borderline.to_csv(DATA / "interim" / "songs_borderline_review.csv", index=False)

print(f"Qualifying (english_ratio >= {CONFIG['min_english_ratio']}): {len(qualifying)} songs by {qualifying['artist_slug'].nunique()} artists")
print(f"Borderline (manual review): {len(borderline)} songs")

Qualifying (english_ratio >= 0.5): 8 songs by 6 artists
Borderline (manual review): 13 songs


---
# Phase 6 — Metadata filtering

In [ ]:
COVER_PATTERNS = [
    r"\(cover\)", r"\[cover\]", r"- cover$",
    r"\(remix\)", r"\[remix\]",
    r"\(live\)", r"\[live\]",
    r"\(acoustic\)", r"\(demo\)", r"\(instrumental\)",
]


def is_likely_cover_or_variant(title):
    if not isinstance(title, str):
        return False
    t = title.lower()
    return any(re.search(p, t) for p in COVER_PATTERNS)


def lyrics_fingerprint(text):
    if not isinstance(text, str):
        return ""
    t = re.sub(r"[^\w\s]", "", text.lower())
    t = re.sub(r"\s+", " ", t).strip()
    return t[:500]


filtered = qualifying.copy()
filtered["has_features"] = filtered["featured_artists"].str.len() > 0
filtered["is_cover_variant"] = filtered["title"].apply(is_likely_cover_or_variant)
filtered["fingerprint"] = filtered["clean_lyrics"].apply(lyrics_fingerprint)

before = len(filtered)
filtered = filtered[~filtered["is_cover_variant"]]
print(f"Dropped covers/variants: {before - len(filtered)} → {len(filtered)} remaining")

before = len(filtered)
filtered = filtered.drop_duplicates(subset=["artist_slug", "fingerprint"])
print(f"Dropped lyrics duplicates: {before - len(filtered)} → {len(filtered)} remaining")

print(f"\nSongs with features (flagged, not dropped): {filtered['has_features'].sum()}")

filtered.to_csv(DATA / "interim" / "songs_filtered.csv", index=False)

---
# Handoff to Phase 7 (manual L1 verification)

Two outputs:
1. **`artists_for_verification.csv`** — one row per artist with metadata, plus blank columns for your manual judgments
2. **`songs_filtered.csv`** — the song-level corpus, automatically filtered

Open the artists CSV, fill in `L1_confirmed`, `decision_include_exclude`, etc., then back-filter songs by your verified artist list.

In [ ]:
artist_summary = (
    filtered.groupby("artist_slug")
    .agg(
        artist_name=("artist_name", "first"),
        n_qualifying_songs=("song_id", "count"),
        n_with_features=("has_features", "sum"),
        mean_english_ratio=("english_ratio", "mean"),
        total_lines=("n_lines", "sum"),
    )
    .reset_index()
    .sort_values("n_qualifying_songs", ascending=False)
)

# Merge in candidate metadata for verification context
candidate_lookup = candidates.copy()
candidate_lookup["artist_slug"] = candidate_lookup["name"].apply(slugify)
artist_summary = artist_summary.merge(
    candidate_lookup[["artist_slug", "birthplace", "birth_year", "genre", "country", "enwiki_url"]],
    on="artist_slug", how="left"
)

artist_summary["meets_song_minimum"] = artist_summary["n_qualifying_songs"] >= CONFIG["min_songs_per_artist"]

# Blank columns for your manual verification
for col in ["L1_confirmed", "raised_in", "age_moved_if_applicable",
            "L1_confidence_high_med_low", "verification_sources",
            "decision_include_exclude", "notes"]:
    artist_summary[col] = ""

artist_summary.to_csv(DATA / "processed" / "artists_for_verification.csv", index=False)
print(f"Artists ready for manual verification: {len(artist_summary)}")
print(f"  Meeting song minimum ({CONFIG['min_songs_per_artist']}+ songs): {artist_summary['meets_song_minimum'].sum()}")
artist_summary.head(15)

In [ ]:
# Pipeline summary
print("=" * 60)
print(f"PIPELINE SUMMARY — L1: {CONFIG['L1']}")
print("=" * 60)
print(f"Phase 2 candidates (merged):     {len(candidates):>6}")
print(f"Phase 3 pre-filter survivors:    {len(pre_filtered):>6}")
print(f"Phase 4 scraped artists:         {songs_raw['artist_slug'].nunique():>6}")
print(f"Phase 4 scraped songs (raw):     {len(songs_raw):>6}")
print(f"Phase 5 english songs:           {len(qualifying):>6}")
print(f"Phase 6 after metadata filter:   {len(filtered):>6}")
print(f"Artists w/ {CONFIG['min_songs_per_artist']}+ qualifying songs:   {artist_summary['meets_song_minimum'].sum():>6}")
print("=" * 60)
print(f"\nNext step: open {DATA / 'processed' / 'artists_for_verification.csv'}")
print("and manually verify L1 for each artist (Phase 7).")